<a href="https://colab.research.google.com/github/Bruno120715/Checkpoint-Merge/blob/main/CheckpointMerge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Configuración de Checkpoint Merge
Define los parámetros para descargar los modelos desde Civitai o enlaces externos, y configura las opciones de fusión (Merge Mode).

In [1]:
# @title Parámetros de Fusión y Descarga

# @markdown ### Autenticación
civitai_api_key = ""  # @param {type:"string"}
hf_token = ""  # @param {type:"string"}

# @markdown ### Enlaces de Modelos (Checkpoints)
checkpoint_link_1 = ""  # @param {type:"string"}
checkpoint_link_2 = ""  # @param {type:"string"}
checkpoint_link_3 = ""  # @param {type:"string"}

# @markdown ### Configuración del Merge
base_model = "Anima"  # @param ["SD 1.5", "SDXL", "Z Image Turbo", "Pony Diffusion V6", "Anima"]
merge_mode = "Weighted Sum"  # @param ["Weighted Sum", "Add Difference", "Sigmoid", "Spline", "Beta"]
calculation_mode = "bf16"  # @param ["fp16", "fp32", "bf16"]
fp16_conversion = False  # @param {type:"boolean"}
alpha = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

import os
import requests

def download_model(url, target_path, api_key=None):
    if not url:
        print("Enlace vacío, omitiendo...")
        return None

    # Si es un enlace de Civitai, añadir el token de API si existe
    if "civitai.com" in url and api_key:
        connector = "&" if "?" in url else "?"
        url = f"{url}{connector}token={api_key}"

    print(f"Descargando modelo desde: {url}")
    os.makedirs(os.path.dirname(target_path), exist_ok=True)

    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(target_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"Guardado en: {target_path}")
        return target_path
    else:
        print(f"Error al descargar: Código de estado {response.status_code}")
        return None

print("Configuración cargada correctamente.")

Configuración cargada correctamente.


In [13]:
# @title 1. Instalar dependencias necesarias
!pip install -q --upgrade safetensors torch pytorch-lightning

In [14]:
# @title 2. Descargar Checkpoints y Ejecutar Fusión (Merge)
import os
import torch
import io
import pickle
from safetensors.torch import load_file, save_file
from huggingface_hub import hf_hub_download, login, list_repo_files

# Definir nombres locales para almacenar los archivos descargados
model_path_1 = "/content/model_1.safetensors"
model_path_2 = "/content/model_2.safetensors"
model_path_3 = "/content/model_3.safetensors"
output_path = "/content/merged_model.safetensors"

# Intentar autenticar en Hugging Face si hay un token disponible
if 'hf_token' in globals() and hf_token:
    try:
        login(token=hf_token)
        print("Autenticado en Hugging Face de forma exitosa.")
    except Exception as e:
        print(f"Advertencia al autenticar en Hugging Face: {e}")

def inteligente_download(link, target_path, api_key=None):
    if not link:
        return None

    # Comprobar si el link parece un identificador de Hugging Face (ej. 'usuario/repo')
    if "/" in link and not link.startswith("http"):
        print(f"Detectado repositorio de Hugging Face: {link}")
        try:
            repo_id = link.strip()
            filename = None

            # Listar todos los archivos del repositorio
            print("Obteniendo la lista de archivos del repositorio...")
            files = list_repo_files(repo_id)

            # Filtrar archivos con extensiones .safetensors o .ckpt en cualquier directorio del repositorio
            weight_files = [f for f in files if f.endswith(".safetensors") or f.endswith(".ckpt")]

            if not weight_files:
                print(f"❌ ERROR: El repositorio '{repo_id}' no contiene checkpoints individuales (.safetensors/.ckpt).")
                print("Nota: Muchos repositorios de Hugging Face usan el formato 'Diffusers' (carpetas separadas).")
                print("Por favor, utiliza un repositorio que contenga un archivo de checkpoint único de Stable Diffusion (Single File).")
                return None

            # Preferir archivos grandes de la raíz o carpetas principales
            safetensors_only = [f for f in weight_files if f.endswith(".safetensors")]
            filename = safetensors_only[0] if safetensors_only else weight_files[0]

            print(f"Descargando '{filename}' desde Hugging Face...")
            descargado = hf_hub_download(repo_id=repo_id, filename=filename, local_dir="/content/", local_dir_use_symlinks=False)

            # Mover o renombrar al target_path esperado
            if os.path.exists(descargado) and descargado != target_path:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                os.replace(descargado, target_path)
            print(f"¡Guardado con éxito en: {target_path}!")
            return target_path
        except Exception as e:
            print(f"Error al descargar de Hugging Face: {e}")
            return None
    else:
        # Si es una URL clásica de Civitai u otra web
        return download_model(link, target_path, api_key)

# 1. Descargar modelos usando el cargador inteligente
print("--- Iniciando descargas de checkpoints ---")
path_1 = inteligente_download(checkpoint_link_1, model_path_1, civitai_api_key) if checkpoint_link_1 else None
path_2 = inteligente_download(checkpoint_link_2, model_path_2, civitai_api_key) if checkpoint_link_2 else None
path_3 = inteligente_download(checkpoint_link_3, model_path_3, civitai_api_key) if checkpoint_link_3 else None

class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if any(x in module for x in ["pytorch_lightning", "lightning", "pydantic"]):
            class Dummy:
                def __init__(self, *args, **kwargs): pass
                def __setstate__(self, state): pass
            return Dummy
        try:
            return super().find_class(module, name)
        except Exception:
            class DummyFallback:
                def __init__(self, *args, **kwargs): pass
                def __setstate__(self, state): pass
            return DummyFallback

def safe_torch_load(filepath):
    """Carga un checkpoint de PyTorch de manera segura usando un Unpickler personalizado."""
    # Creamos un módulo pickle alternativo simulado para pasar a torch.load
    custom_pickle = Object = type('SafePickle', (), {
        'Unpickler': SafeUnpickler,
        'load': lambda f, **kwargs: SafeUnpickler(f, **kwargs).load(),
        'loads': lambda s, **kwargs: SafeUnpickler(io.BytesIO(s), **kwargs).load()
    })

    print("-> Cargando archivo con deserializador seguro...")
    with open(filepath, 'rb') as f:
        buffer = io.BytesIO(f.read())

    return torch.load(buffer, map_location='cpu', pickle_module=custom_pickle)

def load_model_weights(filepath):
    """Carga pesos de forma segura detectando si es PyTorch o Safetensors."""
    if not filepath or not os.path.exists(filepath):
        return None

    with open(filepath, 'rb') as f:
        header = f.read(4)

    if header == b'PK\x03\x04':
        print(f"-> Detectado formato PyTorch (.ckpt) para {filepath}.")
        try:
            checkpoint = safe_torch_load(filepath)
        except Exception as e:
            print(f"Fallo crítico en la carga del archivo PyTorch: {e}")
            raise e

        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            return checkpoint['state_dict']
        return checkpoint
    else:
        print(f"-> Detectado formato Safetensors para {filepath}. Cargando con load_file...")
        return load_file(filepath)

def run_safetensors_merge():
    if not path_1 or not path_2:
        print("Error: Se requieren al menos el Checkpoint 1 y 2 para fusionar y que las descargas hayan sido exitosas.")
        return

    print(f"\n--- Leyendo tensores de los archivos para modelo base {base_model} ---")
    dict_a = load_model_weights(path_1)
    dict_b = load_model_weights(path_2)
    dict_c = load_model_weights(path_3) if ((merge_mode == "Add Difference" or merge_mode == "Beta") and path_3) else None

    if dict_a is None or dict_b is None:
        print("Error: No se pudieron procesar los diccionarios de pesos de los modelos.")
        return

    if calculation_mode == "fp16":
        dtype = torch.float16
    elif calculation_mode == "bf16":
        dtype = torch.bfloat16
    elif calculation_mode == "fp8":
        dtype = torch.float8_e4m3fn if hasattr(torch, 'float8_e4m3fn') else torch.float16
    else:
        dtype = torch.float32

    merged_dict = {}
    print(f"Aplicando fusión: Mode = {merge_mode}, Alpha = {alpha}, Precision = {calculation_mode}")

    keys_a = set(dict_a.keys())
    keys_b = set(dict_b.keys())
    common_keys = keys_a.intersection(keys_b)

    print(f"Total de llaves compartidas encontradas: {len(common_keys)}")

    for key in common_keys:
        t_a = dict_a[key]
        t_b = dict_b[key]

        if not isinstance(t_a, torch.Tensor) or not isinstance(t_b, torch.Tensor):
            continue

        t_a = t_a.to(dtype)
        t_b = t_b.to(dtype)

        if merge_mode == "Weighted Sum":
            merged_tensor = (1.0 - alpha) * t_a + alpha * t_b
        elif merge_mode == "Add Difference" and dict_c is not None and key in dict_c:
            t_c = dict_c[key]
            if isinstance(t_c, torch.Tensor):
                t_c = t_c.to(dtype)
                merged_tensor = t_a + alpha * (t_b - t_c)
            else:
                merged_tensor = t_a
        elif merge_mode == "Beta":
            beta_weight = 3 * (alpha ** 2) - 2 * (alpha ** 3)
            merged_tensor = (1.0 - beta_weight) * t_a + beta_weight * t_b
        else:
            merged_tensor = t_a

        if fp16_conversion:
            merged_tensor = merged_tensor.to(torch.float16)

        merged_dict[key] = merged_tensor

    for key in keys_a - common_keys:
        t_a = dict_a[key]
        if isinstance(t_a, torch.Tensor):
            merged_dict[key] = t_a.to(torch.float16) if fp16_conversion else t_a

    print(f"Guardando resultado final en: {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_file(merged_dict, output_path)
    print("¡El proceso de fusión ha finalizado de manera exitosa!")

try:
    run_safetensors_merge()
except Exception as e:
    print(f"Ocurrió un error inesperado durante el merge: {e}")

--- Iniciando descargas de checkpoints ---
Descargando modelo desde: https://civitai.com/api/download/models/3484735?fileId=3392406&token=7572fd91914ad474c4718de9a68c6c40


KeyboardInterrupt: 

### Guardar Modelo en Google Drive
Usa las siguientes celdas para montar tu Google Drive y transferir el modelo resultante para guardarlo de manera permanente.

In [ ]:
# @title 3. Conectar a Google Drive
from google.colab import drive
import os

try:
    # Intentar montar forzando el remonte en caso de que la carpeta ya esté ocupada
    drive.mount('/content/drive', force_remount=True)
    print("¡Google Drive se ha conectado con éxito!")
except Exception as e:
    print(f"Error al conectar Google Drive: {e}")

In [12]:
# @title 4. Copiar Modelo Fusionado a Google Drive
import shutil
import os

# @markdown Ruta de la carpeta de destino en tu Google Drive
drive_folder = "/content/drive/MyDrive/Ai/Model" # @param {type:"string"}

# @markdown Nombre personalizado para tu archivo final (sin extensión)
custom_model_name = "" # @param {type:"string"}

os.makedirs(drive_folder, exist_ok=True)

source_file = "/content/merged_model.safetensors"

# Asegurar que el nombre tenga la extensión correcta
filename = custom_model_name.strip()
if not filename.endswith(".safetensors"):
    filename += ".safetensors"

target_file = os.path.join(drive_folder, filename)

if os.path.exists(source_file):
    print(f"Copiando {source_file} a {target_file}...")
    shutil.copy(source_file, target_file)
    print("¡Modelo copiado con éxito a tu Google Drive!")
else:
    print("Error: El archivo de modelo fusionado no existe en /content/merged_model.safetensors. Asegúrate de ejecutar la celda de fusión primero.")

Copiando /content/merged_model.safetensors a /content/drive/MyDrive/Ai/Model/Dramazing_v1.5_Chiarezza_Fusion.safetensors...
¡Modelo copiado con éxito a tu Google Drive!


### 5. Instalar y Ejecutar ComfyUI
Utiliza estas celdas para instalar ComfyUI y ejecutarlo directamente en Colab accediendo a tus modelos.

In [15]:
# @title 5.1 Instalar ComfyUI y Dependencias
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -r requirements.txt
!pip install xformers torchsde

# Descargar Cloudflared para exponer la interfaz de forma segura
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("¡ComfyUI instalado con éxito!")

/content
fatal: destination path 'ComfyUI' already exists and is not an empty directory.
/content/ComfyUI
(Reading database ... 127328 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.2) over (2026.11.2) ...
Setting up cloudflared (2026.11.2) ...
Processing triggers for man-db (2.12.0-4build2) ...
¡ComfyUI instalado con éxito!


In [16]:
# @title 5.2 Vincular Modelo Fusionado
import os

# Crear enlace simbólico de nuestro modelo recién fusionado en la carpeta de modelos de ComfyUI
comfy_model_folder = "/content/ComfyUI/models/diffusion_models"
os.makedirs(comfy_model_folder, exist_ok=True)

# Obtener el nombre del modelo y la ruta configurados anteriormente en Google Drive
model_name = globals().get('custom_model_name', 'Neo_Western_Ink_XL').strip()
if not model_name.endswith(".safetensors"):
    model_name += ".safetensors"

drive_folder_path = globals().get('drive_folder', '/content/drive/MyDrive/Ai/Model')

drive_model = os.path.join(drive_folder_path, model_name)
local_model = "/content/merged_model.safetensors"

target_link = os.path.join(comfy_model_folder, "merged_model.safetensors")

# Usamos lexists para detectar y eliminar enlaces rotos o archivos existentes
if os.path.lexists(target_link):
    os.remove(target_link)

if os.path.exists(drive_model):
    os.symlink(drive_model, target_link)
    print(f"¡Se ha vinculado tu modelo guardado en Google Drive ({model_name}) a ComfyUI!")
elif os.path.exists(local_model):
    os.symlink(local_model, target_link)
    print("¡Se ha vinculado el modelo local temporal a ComfyUI!")
else:
    print("No se detectó el modelo fusionado. Asegúrate de haber completado los pasos anteriores.")

¡Se ha vinculado tu modelo guardado en Google Drive (Dramazing_v1.5_Chiarezza_Fusion.safetensors) a ComfyUI!


In [17]:
# @title Descargar VAE y Text Encoders Específicos (Anima & Z Image Turbo) para ComfyUI
import os

vae_dir = "/content/ComfyUI/models/vae"
clip_dir = "/content/ComfyUI/models/clip"

os.makedirs(vae_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)

def download_file(url, target_path):
    # Reemplazar '/blob/' por '/resolve/' si está presente para permitir la descarga directa
    direct_url = url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/resolve/")
    print(f"Descargando: {direct_url} -> {target_path}")
    if not os.path.exists(target_path):
        !wget -q --show-progress -O "{target_path}" "{direct_url}"
    else:
        print("El archivo ya existe de forma local.")

print("--- DESCARGANDO RECURSOS PARA ANIMA ---")
# Anima Text Encoder
anima_te_url = "https://huggingface.co/circlestone-labs/Anima/blob/main/split_files/text_encoders/qwen_3_06b_base.safetensors"
anima_te_path = os.path.join(clip_dir, "qwen_3_06b_base.safetensors")
download_file(anima_te_url, anima_te_path)

# Anima VAE
anima_vae_url = "https://huggingface.co/circlestone-labs/Anima/blob/main/split_files/vae/qwen_image_vae.safetensors"
anima_vae_path = os.path.join(vae_dir, "qwen_image_vae.safetensors")
download_file(anima_vae_url, anima_vae_path)

print("\n--- DESCARGANDO RECURSOS PARA Z IMAGE TURBO ---")
# Z Image Turbo Text Encoder
z_te_url = "https://huggingface.co/Comfy-Org/Z-Image-Turbo/blob/main/split_files/text_encoders/qwen_3_4b.safetensors"
z_te_path = os.path.join(clip_dir, "qwen_3_4b.safetensors")
download_file(z_te_url, z_te_path)

# Z Image Turbo VAE
z_vae_url = "https://huggingface.co/Comfy-Org/Z-Image-Turbo/blob/main/split_files/vae/ae.safetensors"
z_vae_path = os.path.join(vae_dir, "ae.safetensors")
download_file(z_vae_url, z_vae_path)

print("\n¡Todos los Text Encoders y VAEs específicos se han configurado correctamente!")

--- DESCARGANDO RECURSOS PARA ANIMA ---
Descargando: https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors -> /content/ComfyUI/models/clip/qwen_3_06b_base.safetensors
El archivo ya existe de forma local.
Descargando: https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors -> /content/ComfyUI/models/vae/qwen_image_vae.safetensors
El archivo ya existe de forma local.

--- DESCARGANDO RECURSOS PARA Z IMAGE TURBO ---
Descargando: https://huggingface.co/Comfy-Org/Z-Image-Turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors -> /content/ComfyUI/models/clip/qwen_3_4b.safetensors
El archivo ya existe de forma local.
Descargando: https://huggingface.co/Comfy-Org/Z-Image-Turbo/resolve/main/split_files/vae/ae.safetensors -> /content/ComfyUI/models/vae/ae.safetensors
El archivo ya existe de forma local.

¡Todos los Text Encoders y VAEs específicos se han configurado correctamente!


In [18]:
# @title Descargar LoRA Adicional (Anima Turbo LoRA v0.2)
import os

lora_dir = "/content/ComfyUI/models/loras"
os.makedirs(lora_dir, exist_ok=True)

lora_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/loras/anima-turbo-lora-v0.2.safetensors"
lora_path = os.path.join(lora_dir, "anima-turbo-lora-v0.2.safetensors")

print(f"Descargando LoRA desde: {lora_url}")
if not os.path.exists(lora_path):
    !wget -q --show-progress -O "{lora_path}" "{lora_url}"
    print("¡LoRA descargado e instalado correctamente!")
else:
    print("El archivo LoRA ya existe en la carpeta correspondiente.")

Descargando LoRA desde: https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/loras/anima-turbo-lora-v0.2.safetensors
¡LoRA descargado e instalado correctamente!


In [19]:
import subprocess
import threading
import time
import socket
import os

def run_cloudflared():
    # Esperar a que el puerto de ComfyUI responda antes de abrir el túnel
    while True:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        try:
            s.connect(('127.0.0.1', 8188))
            s.close()
            break
        except:
            time.sleep(1)

    print("\n--- Generando enlace de acceso público... ---\n")
    proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if "trycloudflare.com" in line:
            # Extraemos y mostramos claramente la dirección de trycloudflare
            parts = line.split()
            for part in parts:
                if "trycloudflare.com" in part:
                    print(f"\n🔗 ACCEDE A TU COMFYUI AQUÍ: {part.strip()}\n")

# Iniciar túnel de Cloudflare en segundo plano
threading.Thread(target=run_cloudflared, daemon=True).start()

# Asegurar que estamos en el directorio correcto de ComfyUI antes de arrancar
%cd /content/ComfyUI
# Ejecutar ComfyUI con cabeceras CORS habilitadas para evitar bloqueos
!python main.py --listen 127.0.0.1 --port 8188 --enable-cors-header

/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'int8_linear', 'na3d', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'rms_adaln', 'rms_rope', 'rms_rope1', 'rms_rope1_', 'rms_rope_', 'rms_rope_split_half', 'rms_rope_split_half1', 'rms_rope_split_half1_', 'rms